In [ ]:
import json, time, subprocess
from pathlib import Path
import numpy as np, pandas as pd
import onnxruntime as ort
import torch, torch.nn as nn

MODEL = "linear" 
BIT_LEN = "12" #Quantization since zkp can't handle decimals 
DEEPPROVE_PATH = "/root/deep-prove/target/release/bench" #deep prove prover is downloaded in this path inside the linux  

#paths for files
model_dir = Path("../../training") / MODEL
all_sites = sorted(model_dir.glob("site_*"))
assert all_sites, f"no trained sites under {model_dir}" #if they can't find it stop it

work_dir    = Path("_work_dp"); work_dir.mkdir(exist_ok=True)
results_dir = Path("../../results"); results_dir.mkdir(parents=True, exist_ok=True)

def site_id(path): return path.name.replace("site_", "")

#change the windows path to linux path
def to_wsl_path(p):
    p = Path(p).resolve()
    drive = p.drive.rstrip(":").lower()
    return "/mnt/" + drive + "/" + "/".join(p.parts[1:])

print(len(all_sites), "sites available")

321 sites available


In [ ]:
#this function builds a model that has acceptable format for deep prove 
#to rebuild it, we get the weight and bias from the original model and apply them  
def setup_deepprove(site_dir):
    meta = json.load(open(site_dir / "meta.json")) #get the meta data from each site
    seq_len, pred_len = meta["seq_len"], meta["pred_len"] 

    session = ort.InferenceSession(str(site_dir / "network.onnx")) #load the ONNX file from the site

    #get the input name
    input_name = session.get_inputs()[0].name 

    def run(vec):
        x = vec.reshape(1, seq_len, 1).astype(np.float32) #rearrange the numbers into 1 example, 192 hours, and 1 measurement. that's what the model requires
        #run the model and get the output. there's only one output so grab the first one
        return np.array(session.run(None, {input_name: x})[0]).ravel()

    #get the constant bias. the + b part from the equation
    bias = run(np.zeros(seq_len)) #by plugging zeros to the input, we can get what the value of the bias is
    
    #creates an empty 24 x 192 table of zeros so that we can fill the weight from the original model
    #192 past hours and 24 prediction hours (the next day)
    weight = np.zeros((pred_len, seq_len), np.float32)
    
    #activate one 
    for k in range(seq_len):
        zeroInput = np.zeros(seq_len) #fill the empty table with zeros. They are all off 
        #turn on one "hour point" to get the weight of that hour
        #that way we can see how much that one hour affect the output and with simple calculation, we can get the weight of that
        zeroInput[k] = 1.0 
        weight[:, k] = run(zeroInput) - bias #calculate the actual output  

    #double check the weight and the bias. compare the formular we made matches with the formula from the original model
    input_data = np.array(json.load(open(site_dir / "input.json"))["input_data"][0], np.float32)
    
    #if the error is very small like less than 0.000001, we can say that it's correct
    error = float(np.abs((weight @ input_data + bias) - run(input_data)).max())

    #make the multiply-and-add layer 
    #but pytorch automatically fills with random weights
    fused = nn.Linear(seq_len, pred_len) 

    #so overwrite the weights and the bias that was extracted earlier
    with torch.no_grad():
        fused.weight.copy_(torch.from_numpy(weight)) #copy the weight from weight
        fused.bias.copy_(torch.from_numpy(bias)) 

    #make it ready for predicting and not training
    fused.eval()

    fused_dir = Path("../../fuse_models") / MODEL / f"site_{site_id(site_dir)}"
    fused_dir.mkdir(parents=True, exist_ok=True)
    

    #save the new model and all sites
    onnx_path  = fused_dir / "network.onnx"
    torch.onnx.export(fused, torch.zeros(1, seq_len), str(onnx_path),
                      input_names=["input"], output_names=["output"],
                      opset_version=13, dynamo=False)

    #deep prove only accepts number from -1 to 1. so the numbers should be scaled properly to fit in that range
    max_number = float(np.max(np.abs(input_data))) or 1.0
    
    #find the maximum number and divide everything by that number. That way the biggest number will be 1 or -1
    scaled_value = (input_data / max_number).astype(np.float32)

    #calculate the ouput using the new scaled number
    scaled_out = (weight @ scaled_value + bias).astype(np.float32)

    input_path = fused_dir / "input_dp.json"
    
    #record the result
    json.dump({"input_data": [scaled_value.tolist()], "output_data": [scaled_out.tolist()],
               "pytorch_output": [scaled_out.tolist()]}, open(input_path, "w"))
    
    return onnx_path, input_path, meta, error

In [ ]:
#just a simple validation and make sure all sites are selected
sites = all_sites
print(len(sites), "selected")

321 selected


In [ ]:
#now run the prove using the fused model
def prove_site(onnx_path, input_path, csv_path):
    cmd = ["wsl.exe", "-d", "Ubuntu", "-u", "root", "--",
           "env", f"ZKML_BIT_LEN={BIT_LEN}", DEEPPROVE_PATH,
           "-o", to_wsl_path(onnx_path), "-i", to_wsl_path(input_path),
           "--bench", to_wsl_path(csv_path), "--num-samples", "1"]
    
    start = time.perf_counter()

    result = subprocess.run(cmd, capture_output=True, text=True)
    total_time = time.perf_counter() - start

    #check if it worked or not
    if result.returncode != 0:
        print(result.stdout[-1500:]); print(result.stderr[-3000:])
        raise RuntimeError(f"bench failed rc={result.returncode}")
    return total_time


rows = []

#run the prove for each site 
for site_dir in sites:
    onnx_path, input_path, meta, error = setup_deepprove(site_dir) #build the fuse model for this site
    csv_path = work_dir / f"bench_{site_id(site_dir)}.csv"

    if csv_path.exists(): csv_path.unlink()
    total_time = prove_site(onnx_path, input_path, csv_path) 

    bench_df = pd.read_csv(csv_path).set_index("name")

    #format the outpu
    def metric(name, col="elapsed"):
        return float(bench_df.loc[name, col]) if name in bench_df.index else None
    
    prove_ms, verify_ms = metric("Proving 0"), metric("Verify 0")

    rows.append({
        "framework": "deepprove", "model": MODEL, "site": meta["site"],
        "params": meta["params"], "bit_len": int(BIT_LEN),
        "prove_s": round(prove_ms / 1000, 4) if prove_ms is not None else None,
        "verify_s": round(verify_ms / 1000, 4) if verify_ms is not None else None,
        "inference_s": round((metric("Inference 0") or 0) / 1000, 4),
        "context_s": round((metric("Context creation") or 0) / 1000, 4),
        "proof_kb": metric("Proving 0", "proof_size"),
        "peak_mem_mb": round((metric("Proving 0", "peak") or 0) / 1024**2, 1),
        "total_time": round(total_time, 3),
        "affine_recon_err": error,
        "mse_float": meta["mse_float"],
        "verified": verify_ms is not None,   # bench aborts on a bad proof, so a Verify row means valid
    })
    print(f"site {site_id(site_dir):>3}: prove={rows[-1]['prove_s']}s verify={rows[-1]['verify_s']}s "
          f"proof={rows[-1]['proof_kb']}KB recon_err={error:.1e}")

In [5]:
results = pd.DataFrame(rows)
out_path = results_dir / f"deepprove_{MODEL}.csv"
results.to_csv(out_path, index=False)
print("wrote", out_path, "(", len(results), "rows )")
results[["prove_s", "verify_s", "context_s", "proof_kb", "peak_mem_mb"]].describe().round(3)

wrote ..\..\results\deepprove_linear.csv ( 321 rows )


,prove_s,verify_s,context_s,proof_kb,peak_mem_mb
count,321.000,321.000,321.000,321.000,321.000
mean,0.929,0.018,0.397,40.033,4173.412
std,0.135,0.002,0.060,0.000,15.132
min,0.522,0.008,0.222,40.033,3918.600
25%,0.840,0.017,0.360,40.033,4174.600
50%,0.876,0.018,0.380,40.033,4174.600
75%,1.004,0.019,0.418,40.033,4174.600
max,1.612,0.040,0.700,40.033,4174.800
